This notebook generates all plots regarding the evaluation of therapies (BEMPEG + model-optimized IL-2) in the tumor microenvironment of metastatic melanoma.

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import use
from scipy.stats import spearmanr,pearsonr
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
from collections import Counter
from math import floor

In [4]:
# Load BEMPEG data simulating CTLL-2 and immune cell types
df_sim = pd.read_csv("results/BEMPEG_trials/simulations_BEMPEG_ABC.csv")
df_sim_data = pd.read_csv('data/signaling/BEMPEG_data.csv')
# Colors for mutants:
color_sim = ["darkred","blue","darkgreen"]
color_invit = ["salmon","cornflowerblue","forestgreen"]
markers = ['*','o','^']
linestyles = ["dashed","dotted","dashdot"]

### Plot dose-response curves of BEMPEG variants

In [4]:
# First plot CTLL-2
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_cell = df_sim.loc[df_sim["Cell_type"]=="CTLL-2"]
i = 0
max_pSTAT5 = df_cell.loc[df_cell["Variant"]=="WT","Result"].max()
for variant in df_cell["Variant"].drop_duplicates():
    ax.plot(df_cell.loc[df_cell["Variant"]==variant,"IL0"], df_cell.loc[df_cell["Variant"]==variant,"Result"].values/max_pSTAT5*100, linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
    ax.scatter(df_sim_data.loc[df_sim_data["Variant"]==variant,"IL"], df_sim_data.loc[df_sim_data["Variant"]==variant,"pSTAT"], s=200, linewidth=2, color=color_invit[i], marker=markers[i])
    i += 1
plt.legend(loc="upper left", fontsize=20)
plt.xscale('log')
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.ylabel('pSTAT5 (%)', fontsize=25)
plt.xlabel('IL-2 (M)', fontsize=25)
plt.title("CTLL-2", fontsize=25)
plt.savefig('figures/BEMPEG_trials/Plot_BEMPEG_CTLL-2.pdf', transparent=True, bbox_inches="tight")
plt.close()

In [12]:
# Plot other cells
for cell in df_sim.loc[df_sim["Cell_type"]!="CTLL-2","Cell_type"].drop_duplicates():
    df_cell = df_sim.loc[df_sim["Cell_type"]==cell]
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    i = 0
    max_pSTAT5 = df_cell.loc[df_cell["Variant"]=="WT","Result"].max()
    for variant in df_cell["Variant"].drop_duplicates():
        ax.plot(df_cell.loc[df_cell["Variant"]==variant,"IL0"], df_cell.loc[df_cell["Variant"]==variant,"Result"].values/max_pSTAT5*100, linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
        i += 1
    plt.legend(loc="upper left", fontsize=20)
    plt.xscale('log')
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.ylabel('pSTAT5 (%)', fontsize=25)
    plt.xlabel('IL-2 (M)', fontsize=25)
    if cell == "Treg":
        cell = "Regulatory T cells"
    elif cell == "T8":
        cell = "CD8+ T cells"
    plt.title(cell, fontsize=25)
    plt.savefig('figures/BEMPEG_trials/Plot_BEMPEG_'+cell+'.pdf', transparent=True, bbox_inches="tight")
    plt.close()

### BEMPEG simulations on melanoma tumor data

In [14]:
# Load data and separate it on 3 groups, non-IL2 target cells and IL2 target activated and unactivated cells
df = pd.read_csv("results/BEMPEG_trials/simulations_BEMPEG_tumor_IL2s.csv")
df_high = df.loc[df["IL0_1"]==1e-4][["Cell_type","RB0","STAT50","Result"]]
df_treatment = df.loc[df["IL0_1"]!=1e-4][["Cell_type","RB0","STAT50","Result"]]
# Separate cells into 3 groups
df_treatment["Abs. activ."] = df_treatment["Result"].values
df_treatment["Activ. pot."] = df_treatment["Result"].values/df_high["Result"].values*100
df_treatment = df_treatment[["Cell_type","RB0","Abs. activ.","Activ. pot."]]
thres = 40 #(df_treatment[df_treatment["Abs. activ."]>df_treatment["Abs. activ."].mean()]["Activ. pot."].mean() // 10) * 10 # As threshold we take the mean activ. potential of 50% of cells with more pSTAT5
df_target = df_treatment.loc[df_treatment["RB0"]>2].loc[df_treatment.loc[df_treatment["RB0"]>2]["Activ. pot."]>thres]
df_notarget = df_treatment.loc[df_treatment["RB0"]>2].loc[df_treatment.loc[df_treatment["RB0"]>2]["Activ. pot."]<=thres]

In [17]:
# Make scatter plot
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
ax.scatter(df_treatment["Abs. activ."].values, df_treatment["Activ. pot."].values, s=80, linewidth=0.8, color="darkgreen",marker='o',alpha = 0.33, label="Non-IL2 target cells")
ax.scatter(df_target["Abs. activ."].values, df_target["Activ. pot."].values, s=80, linewidth=0.8, color="darkred",marker='o',alpha = 0.33,label="IL2 target activated cells")
ax.scatter(df_notarget["Abs. activ."].values, df_notarget["Activ. pot."].values, s=80, linewidth=0.8, color="darkblue",marker='o',alpha = 0.33,label="IL2 target inactivated cells")
plt.legend(loc="upper left", fontsize=20)
plt.xscale("log")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("Predicted BEMPEG response", fontsize=25)
plt.xlabel('Absolute activation (pSTAT5 num.)', fontsize=25)
plt.ylabel('Activation potential (% pSTAT5)', fontsize=25)
plt.savefig('figures/BEMPEG_trials/Plot_tumor.pdf', transparent=True, bbox_inches="tight")
plt.close()

In [22]:
# Get percentages of cells in IL2 target, activated cells
perc_cells = Counter([cell.split("__")[0] for cell in df_treatment["Cell_type"].values])
df = pd.DataFrame()
df["Cell"] = perc_cells.keys()
df["% total cells"] = np.array(list(perc_cells.values()))/len(df_treatment["Cell_type"].values)*100
perc_cells = Counter([cell.split("__")[0] for cell in df_target["Cell_type"].values])
df2 = pd.DataFrame()
df2["Cell"] = perc_cells.keys()
df2["% target cells"] = np.array(list(perc_cells.values()))/len(df_target["Cell_type"].values)*100
df_tot_tar = pd.merge(df, df2, on='Cell', how='outer')
df_tot_tar = df_tot_tar.fillna(0)

# Get percentages of cells in IL2 target, non-activated cells
perc_cells = Counter([cell.split("__")[0] for cell in df_notarget["Cell_type"].values])
df3 = pd.DataFrame()
df3["Cell"] = perc_cells.keys()
df3["% non-target cells"] = np.array(list(perc_cells.values()))/len(df_notarget["Cell_type"].values)*100
df_tot_ntar = pd.merge(df, df3, on='Cell', how='outer')
df_tot_ntar = df_tot_ntar.fillna(0)

In [23]:
# Sample from the full population to get error bars in IL2 target, activated cells 
perc_cells = Counter([cell.split("__")[0] for cell in df_target["Cell_type"].values])
df_t = pd.DataFrame()
df_t["Cell"] = perc_cells.keys()
for i in range(0,250):
    df_target_s = df_target.sample(n=int(df_target.index.size/2))
    perc_cells = Counter([cell.split("__")[0] for cell in df_target_s["Cell_type"].values])
    df_ts = pd.DataFrame()
    df_ts["Cell"] = perc_cells.keys()
    df_ts["% "+str(i)] = np.array(list(perc_cells.values()))/len(df_target_s["Cell_type"].values)*100
    df_t = pd.merge(df_t, df_ts, on='Cell', how='outer')
df_t.set_index('Cell', inplace=True)
df_t = df_t.fillna(0)
df_t["95_conf"] = 1.96*(df_t.var(axis=1)/np.sqrt(len(df_t.columns)))
df_t = df_t[["95_conf"]]
df_tot_tar = pd.merge(df_t, df_tot_tar, on='Cell', how='outer')
df_tot_tar = df_tot_tar.fillna(0)

# Sample from the full population to get error bars in IL2 target, non-activated cells 
perc_cells = Counter([cell.split("__")[0] for cell in df_notarget["Cell_type"].values])
df_nt = pd.DataFrame()
df_nt["Cell"] = perc_cells.keys()
for i in range(0,250):
    df_notarget_s = df_notarget.sample(n=int(df_notarget.index.size/2))
    perc_cells = Counter([cell.split("__")[0] for cell in df_notarget_s["Cell_type"].values])
    df_nts = pd.DataFrame()
    df_nts["Cell"] = perc_cells.keys()
    df_nts["% "+str(i)] = np.array(list(perc_cells.values()))/len(df_notarget_s["Cell_type"].values)*100
    df_nt = pd.merge(df_nt, df_nts, on='Cell', how='outer')
df_nt.set_index('Cell', inplace=True)
df_nt = df_nt.fillna(0)
df_nt["95_conf"] = 1.96*(df_nt.var(axis=1)/np.sqrt(len(df_nt.columns)))
df_nt = df_nt[["95_conf"]]
df_tot_ntar = pd.merge(df_nt, df_tot_ntar, on='Cell', how='outer')
df_tot_ntar = df_tot_ntar.fillna(0)

In [24]:
# Get all plots to have the same order of cells
new_cell_order = ['CD4_naive', 'CD4_Central_memory', 'CD4_Effector_memory', 'CD8_Central_memory_cytotoxic', 'CD8_Effector_memory', 'CD8_Exhausted', 'CD8_Exhausted_cytotoxic', 'Classical_Mono', 'ILC1', 'Macrophage_M1', 'NK_bright', 'Tfh', 'Th17', 'Th1', 'Th2', 'Treg_PD1+', 'Treg', 'cDC', 'CD8_Central_memory', 'CD8_Cytotoxic', 'CD8_Effector_memory_cytotoxic', 'CD8_naive', 'ILC2', 'ILC3', 'Macrophage_M2', 'Memory_B', 'NK_dim', 'Naive_B', 'NonClassical_Mono', 'Plasma_Cell', 'pDC', 'ILC']
df_tot_ntar['Cell'] = pd.Categorical(df_tot_ntar['Cell'], categories=new_cell_order, ordered=True)
df_tot_ntar = df_tot_ntar.sort_values('Cell')
df_tot_tar['Cell'] = pd.Categorical(df_tot_tar['Cell'], categories=new_cell_order, ordered=True)
df_tot_tar = df_tot_tar.sort_values('Cell')

# Make barplots of IL2 target activated and non-activated cells
fig,ax=plt.subplots(1,1,figsize=(14, 8), dpi=400)
plt.bar(df_tot_tar["Cell"], np.zeros([1,len(df_tot_tar["Cell"])])[0], width=0.9, alpha=0.8, color="darkred", label="IL2 target activated cells")
plt.bar(df_tot_ntar["Cell"], np.zeros([1,len(df_tot_tar["Cell"])])[0], width=0.9, alpha=0.8, color="darkblue", label="IL2 target inactivated cells")
for cell in df_tot_tar["Cell"]:
    if df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"].values[0] < df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"].values[0]:
        plt.bar(cell, df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"], width=0.9, alpha=0.8, color="darkblue")
        plt.bar(cell, df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"], width=0.9, alpha=0.8, color="darkred")
    else:
        plt.bar(cell, df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"], width=0.9, alpha=0.8, color="darkred")
        plt.bar(cell, df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"], width=0.9, alpha=0.8, color="darkblue")
plt.errorbar(df_tot_tar["Cell"], df_tot_tar["% target cells"], df_tot_tar["95_conf"], fmt='none', color='black' , ecolor='black', elinewidth=5, capsize=7, barsabove=True, capthick=3)
plt.errorbar(df_tot_ntar["Cell"], df_tot_ntar["% non-target cells"], df_tot_ntar["95_conf"], fmt='none', color='black' , ecolor='black', elinewidth=5, capsize=7, barsabove=True, capthick=3)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=15)
plt.ylabel('Cell proportion (%)', fontsize=20)
plt.xticks(rotation=90, ha="right")
ax.legend(loc="upper right", fontsize=15)
plt.title("Predicted BEMPEG response", fontsize=20)
plt.savefig('figures/BEMPEG_trials/Perc_cells_all_tumor.pdf', bbox_inches='tight', transparent=True)
plt.close()

#### RB=100 expression limit plot

In [25]:
# Load data
df = pd.read_csv("results/BEMPEG_trials/simulations_BEMPEG_heat_map.csv")
df_high = df.loc[df["IL0_1"]==1e-4][["Cell_type","RB0","STAT50","Result"]]

# Plot
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
ax.plot(df_high["RB0"][1:], df_high["Result"][1:], linewidth=7.5, color="darkred", linestyle="dashed")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.ylabel('Absolute pSTAT5', fontsize=25)
plt.xlabel('log10(IL-2RB)', fontsize=25)
plt.savefig('figures/BEMPEG_trials/Plot_RB_limit.pdf', transparent=True, bbox_inches="tight")
plt.close()

### Plot dose-response curves of the model-optimized mutant

In [5]:
# Load optimized variant data simulating target and non_target immune cell types
df_sim = pd.read_csv("results/BEMPEG_trials/simulations_BEMPEG_ABC_optimized.csv")
# Colors for mutants:
color_sim = ["darkred","blue","darkgreen","darkorange","purple","gold","springgreen","cyan"]
linestyles = ["dashed","dotted","dashdot",(5, (10, 3)),(0, (5, 10)),(0, (3, 1, 1, 1, 1, 1)),(0, (3, 1, 1, 1)),"solid"]

In [6]:
df_sim = df_sim.loc[df_sim["Variant"].isin(["WT","Mut_full"])]
# Plot full and partial proposed mutants + WT
for cell in df_sim["Cell_type"].drop_duplicates():
    df_cell = df_sim.loc[df_sim["Cell_type"]==cell]
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    i = 0
    max_pSTAT5 = df_cell.loc[df_cell["Variant"]=="WT","Result"].max()
    for variant in df_cell["Variant"].drop_duplicates():
        ax.plot(df_cell.loc[df_cell["Variant"]==variant,"IL0"], df_cell.loc[df_cell["Variant"]==variant,"Result"].values/max_pSTAT5*100, linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
        i += 1
    plt.legend(loc="upper left", fontsize=20)
    plt.xscale('log')
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.ylabel('pSTAT5 (%)', fontsize=25)
    plt.xlabel('IL-2 (M)', fontsize=25)
    if cell == "Treg":
        cell = "Regulatory T cells"
    elif cell == "T8":
        cell = "CD8+ T cells"
    plt.title(cell, fontsize=25)
    plt.savefig('figures/BEMPEG_trials/optimized/Plot_BEMPEG_'+cell+'.pdf', transparent=True, bbox_inches="tight")
    plt.close()

### Model-optimized mutant simulations on melanoma tumor data

In [7]:
# Load data and separate it on 3 groups, non-IL2 target cells and IL2 target activated and unactivated cells
df = pd.read_csv("results/BEMPEG_trials/simulations_BEMPEG_tumor_full_optimized.csv")
df_high = df.loc[df["IL0"]==1e-4][["Cell_type","RB0","STAT50","Result"]]
df_treatment = df.loc[df["IL0"]!=1e-4][["Cell_type","RB0","STAT50","Result"]]
# Separate cells into 3 groups
df_treatment["Abs. activ."] = df_treatment["Result"].values
df_treatment["Activ. pot."] = df_treatment["Result"].values/df_high["Result"].values*100
df_treatment = df_treatment[["Cell_type","RB0","Abs. activ.","Activ. pot."]]
thres = 40 
df_target = df_treatment.loc[df_treatment["RB0"]>2].loc[df_treatment.loc[df_treatment["RB0"]>2]["Activ. pot."]>thres]
df_notarget = df_treatment.loc[df_treatment["RB0"]>2].loc[df_treatment.loc[df_treatment["RB0"]>2]["Activ. pot."]<=thres]

In [9]:
# Make scatter plot
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
ax.scatter(df_treatment["Abs. activ."].values, df_treatment["Activ. pot."].values, s=80, linewidth=0.8, color="darkgreen",marker='o',alpha = 0.33, label="Non-IL2 target cells")
ax.scatter(df_target["Abs. activ."].values, df_target["Activ. pot."].values, s=80, linewidth=0.8, color="darkred",marker='o',alpha = 0.33,label="IL2 target activated cells")
ax.scatter(df_notarget["Abs. activ."].values, df_notarget["Activ. pot."].values, s=80, linewidth=0.8, color="darkblue",marker='o',alpha = 0.33,label="IL2 target inactivated cells")
plt.legend(loc="lower left", fontsize=20)
plt.xscale("log")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
# ax.set_xticks([0.01,0.1,1,10,100,1000])
plt.title("Predicted mutant response", fontsize=25)
plt.xlabel('Absolute activation (pSTAT5 num.)', fontsize=25)
plt.ylabel('Activation potential (% pSTAT5)', fontsize=25)
plt.savefig('figures/BEMPEG_trials/optimized/Plot_tumor_optimized.pdf', transparent=True, bbox_inches="tight")
plt.close()

In [10]:
# Get percentages of cells in IL2 target, activated cells
perc_cells = Counter([cell.split("__")[0] for cell in df_treatment["Cell_type"].values])
df = pd.DataFrame()
df["Cell"] = perc_cells.keys()
df["% total cells"] = np.array(list(perc_cells.values()))/len(df_treatment["Cell_type"].values)*100
perc_cells = Counter([cell.split("__")[0] for cell in df_target["Cell_type"].values])
df2 = pd.DataFrame()
df2["Cell"] = perc_cells.keys()
df2["% target cells"] = np.array(list(perc_cells.values()))/len(df_target["Cell_type"].values)*100
df_tot_tar = pd.merge(df, df2, on='Cell', how='outer')
df_tot_tar = df_tot_tar.fillna(0)

# Get percentages of cells in IL2 target, non-activated cells
perc_cells = Counter([cell.split("__")[0] for cell in df_notarget["Cell_type"].values])
df3 = pd.DataFrame()
df3["Cell"] = perc_cells.keys()
df3["% non-target cells"] = np.array(list(perc_cells.values()))/len(df_notarget["Cell_type"].values)*100
df_tot_ntar = pd.merge(df, df3, on='Cell', how='outer')
df_tot_ntar = df_tot_ntar.fillna(0)

In [11]:
# Sample from the full population to get error bars in IL2 target, activated cells 
perc_cells = Counter([cell.split("__")[0] for cell in df_target["Cell_type"].values])
df_t = pd.DataFrame()
df_t["Cell"] = perc_cells.keys()
for i in range(0,250):
    df_target_s = df_target.sample(n=int(df_target.index.size/2))
    perc_cells = Counter([cell.split("__")[0] for cell in df_target_s["Cell_type"].values])
    df_ts = pd.DataFrame()
    df_ts["Cell"] = perc_cells.keys()
    df_ts["% "+str(i)] = np.array(list(perc_cells.values()))/len(df_target_s["Cell_type"].values)*100
    df_t = pd.merge(df_t, df_ts, on='Cell', how='outer')
df_t.set_index('Cell', inplace=True)
df_t = df_t.fillna(0)
df_t["95_conf"] = 1.96*(df_t.var(axis=1)/np.sqrt(len(df_t.columns)))
df_t = df_t[["95_conf"]]
df_tot_tar = pd.merge(df_t, df_tot_tar, on='Cell', how='outer')
df_tot_tar = df_tot_tar.fillna(0)

# Sample from the full population to get error bars in IL2 target, non-activated cells 
perc_cells = Counter([cell.split("__")[0] for cell in df_notarget["Cell_type"].values])
df_nt = pd.DataFrame()
df_nt["Cell"] = perc_cells.keys()
for i in range(0,250):
    df_notarget_s = df_notarget.sample(n=int(df_notarget.index.size/2))
    perc_cells = Counter([cell.split("__")[0] for cell in df_notarget_s["Cell_type"].values])
    df_nts = pd.DataFrame()
    df_nts["Cell"] = perc_cells.keys()
    df_nts["% "+str(i)] = np.array(list(perc_cells.values()))/len(df_notarget_s["Cell_type"].values)*100
    df_nt = pd.merge(df_nt, df_nts, on='Cell', how='outer')
df_nt.set_index('Cell', inplace=True)
df_nt = df_nt.fillna(0)
df_nt["95_conf"] = 1.96*(df_nt.var(axis=1)/np.sqrt(len(df_nt.columns)))
df_nt = df_nt[["95_conf"]]
df_tot_ntar = pd.merge(df_nt, df_tot_ntar, on='Cell', how='outer')
df_tot_ntar = df_tot_ntar.fillna(0)

In [13]:
new_cell_order = ['CD4_naive', 'CD4_Central_memory', 'CD4_Effector_memory', 'CD8_Central_memory_cytotoxic', 'CD8_Effector_memory', 'CD8_Exhausted', 'CD8_Exhausted_cytotoxic', 'Classical_Mono', 'ILC1', 'Macrophage_M1', 'NK_bright', 'Tfh', 'Th17', 'Th1', 'Th2', 'Treg_PD1+', 'Treg', 'cDC', 'CD8_Central_memory', 'CD8_Cytotoxic', 'CD8_Effector_memory_cytotoxic', 'CD8_naive', 'ILC2', 'ILC3', 'Macrophage_M2', 'Memory_B', 'NK_dim', 'Naive_B', 'NonClassical_Mono', 'Plasma_Cell', 'pDC', 'ILC']
# Get all plots to have the same order of cells
df_tot_ntar['Cell'] = pd.Categorical(df_tot_ntar['Cell'], categories=new_cell_order, ordered=True)
df_tot_ntar = df_tot_ntar.sort_values('Cell')
df_tot_tar['Cell'] = pd.Categorical(df_tot_tar['Cell'], categories=new_cell_order, ordered=True)
df_tot_tar = df_tot_tar.sort_values('Cell')

# Make barplots of IL2 target activated and non-activated cells
fig,ax=plt.subplots(1,1,figsize=(14, 8), dpi=400)
plt.bar(df_tot_tar["Cell"], df_tot_tar["% target cells"], width=0.9, color="darkred", alpha=0.8, label="IL2 target activated cells")
plt.errorbar(df_tot_tar["Cell"], df_tot_tar["% target cells"], df_tot_tar["95_conf"], fmt='none', color='black' , ecolor='black', elinewidth=5, capsize=7, barsabove=True, capthick=3)
plt.bar(df_tot_ntar["Cell"], df_tot_ntar["% non-target cells"], capsize=7, width=0.9, color="darkblue", alpha=0.8,label="IL2 target inactivated cells")
plt.errorbar(df_tot_ntar["Cell"], df_tot_ntar["% non-target cells"], df_tot_ntar["95_conf"], fmt='none', color='black' , ecolor='black', elinewidth=5, capsize=7, barsabove=True, capthick=3)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=15)
plt.ylabel('Cell proportion (%)', fontsize=20)
plt.xticks(rotation=90, ha="right")
ax.legend(loc="upper right", fontsize=15)
plt.title("Predicted mutant response", fontsize=20)
plt.savefig('figures/BEMPEG_trials/optimized/Perc_cells_all_tumor_optimized.pdf', bbox_inches='tight', transparent=True)
plt.close()

#### Get response of BEMPEG in a concentration 10 times less or more than the one calculated from the paper

In [5]:
# Load data and separate it on 3 groups, non-IL2 target cells and IL2 target activated and unactivated cells
df = pd.read_csv("results/BEMPEG_trials/simulations_BEMPEG_tumor_IL2s_x10.csv")
df_high = df.loc[df["IL0_1"]==1e-4][["Cell_type","RB0","STAT50","Result"]]
df_treatment = df.loc[df["IL0_1"]!=1e-4][["Cell_type","RB0","STAT50","Result"]]
# Separate cells into 3 groups
df_treatment["Abs. activ."] = df_treatment["Result"].values
df_treatment["Activ. pot."] = df_treatment["Result"].values/df_high["Result"].values*100
df_treatment = df_treatment[["Cell_type","RB0","Abs. activ.","Activ. pot."]]
thres = 40 #(df_treatment[df_treatment["Abs. activ."]>df_treatment["Abs. activ."].mean()]["Activ. pot."].mean() // 10) * 10 # As threshold we take the mean activ. potential of 50% of cells with more pSTAT5
df_target = df_treatment.loc[df_treatment["RB0"]>2].loc[df_treatment.loc[df_treatment["RB0"]>2]["Activ. pot."]>thres]
df_notarget = df_treatment.loc[df_treatment["RB0"]>2].loc[df_treatment.loc[df_treatment["RB0"]>2]["Activ. pot."]<=thres]

In [6]:
# Make scatter plot
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
ax.scatter(df_treatment["Abs. activ."].values, df_treatment["Activ. pot."].values, s=80, linewidth=0.8, color="darkgreen",marker='o',alpha = 0.33, label="Non-IL2 target cells")
ax.scatter(df_target["Abs. activ."].values, df_target["Activ. pot."].values, s=80, linewidth=0.8, color="darkred",marker='o',alpha = 0.33,label="IL2 target activated cells")
ax.scatter(df_notarget["Abs. activ."].values, df_notarget["Activ. pot."].values, s=80, linewidth=0.8, color="darkblue",marker='o',alpha = 0.33,label="IL2 target inactivated cells")
plt.legend(loc="upper left", fontsize=20)
plt.xscale("log")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("Predicted BEMPEG response", fontsize=25)
plt.xlabel('Absolute activation (pSTAT5 num.)', fontsize=25)
plt.ylabel('Activation potential (% pSTAT5)', fontsize=25)
plt.savefig('figures/BEMPEG_trials/Plot_tumor_x10.pdf', transparent=True, bbox_inches="tight")
plt.close()

In [7]:
# Get percentages of cells in IL2 target, activated cells
perc_cells = Counter([cell.split("__")[0] for cell in df_treatment["Cell_type"].values])
df = pd.DataFrame()
df["Cell"] = perc_cells.keys()
df["% total cells"] = np.array(list(perc_cells.values()))/len(df_treatment["Cell_type"].values)*100
perc_cells = Counter([cell.split("__")[0] for cell in df_target["Cell_type"].values])
df2 = pd.DataFrame()
df2["Cell"] = perc_cells.keys()
df2["% target cells"] = np.array(list(perc_cells.values()))/len(df_target["Cell_type"].values)*100
df_tot_tar = pd.merge(df, df2, on='Cell', how='outer')
df_tot_tar = df_tot_tar.fillna(0)

# Get percentages of cells in IL2 target, non-activated cells
perc_cells = Counter([cell.split("__")[0] for cell in df_notarget["Cell_type"].values])
df3 = pd.DataFrame()
df3["Cell"] = perc_cells.keys()
df3["% non-target cells"] = np.array(list(perc_cells.values()))/len(df_notarget["Cell_type"].values)*100
df_tot_ntar = pd.merge(df, df3, on='Cell', how='outer')
df_tot_ntar = df_tot_ntar.fillna(0)

In [8]:
# Sample from the full population to get error bars in IL2 target, activated cells 
perc_cells = Counter([cell.split("__")[0] for cell in df_target["Cell_type"].values])
df_t = pd.DataFrame()
df_t["Cell"] = perc_cells.keys()
for i in range(0,250):
    df_target_s = df_target.sample(n=int(df_target.index.size/2))
    perc_cells = Counter([cell.split("__")[0] for cell in df_target_s["Cell_type"].values])
    df_ts = pd.DataFrame()
    df_ts["Cell"] = perc_cells.keys()
    df_ts["% "+str(i)] = np.array(list(perc_cells.values()))/len(df_target_s["Cell_type"].values)*100
    df_t = pd.merge(df_t, df_ts, on='Cell', how='outer')
df_t.set_index('Cell', inplace=True)
df_t = df_t.fillna(0)
df_t["95_conf"] = 1.96*(df_t.var(axis=1)/np.sqrt(len(df_t.columns)))
df_t = df_t[["95_conf"]]
df_tot_tar = pd.merge(df_t, df_tot_tar, on='Cell', how='outer')
df_tot_tar = df_tot_tar.fillna(0)

# Sample from the full population to get error bars in IL2 target, non-activated cells 
perc_cells = Counter([cell.split("__")[0] for cell in df_notarget["Cell_type"].values])
df_nt = pd.DataFrame()
df_nt["Cell"] = perc_cells.keys()
for i in range(0,250):
    df_notarget_s = df_notarget.sample(n=int(df_notarget.index.size/2))
    perc_cells = Counter([cell.split("__")[0] for cell in df_notarget_s["Cell_type"].values])
    df_nts = pd.DataFrame()
    df_nts["Cell"] = perc_cells.keys()
    df_nts["% "+str(i)] = np.array(list(perc_cells.values()))/len(df_notarget_s["Cell_type"].values)*100
    df_nt = pd.merge(df_nt, df_nts, on='Cell', how='outer')
df_nt.set_index('Cell', inplace=True)
df_nt = df_nt.fillna(0)
df_nt["95_conf"] = 1.96*(df_nt.var(axis=1)/np.sqrt(len(df_nt.columns)))
df_nt = df_nt[["95_conf"]]
df_tot_ntar = pd.merge(df_nt, df_tot_ntar, on='Cell', how='outer')
df_tot_ntar = df_tot_ntar.fillna(0)

In [9]:
# Get all plots to have the same order of cells
new_cell_order = ['CD4_naive', 'CD4_Central_memory', 'CD4_Effector_memory', 'CD8_Central_memory_cytotoxic', 'CD8_Effector_memory', 'CD8_Exhausted', 'CD8_Exhausted_cytotoxic', 'Classical_Mono', 'ILC1', 'Macrophage_M1', 'NK_bright', 'Tfh', 'Th17', 'Th1', 'Th2', 'Treg_PD1+', 'Treg', 'cDC', 'CD8_Central_memory', 'CD8_Cytotoxic', 'CD8_Effector_memory_cytotoxic', 'CD8_naive', 'ILC2', 'ILC3', 'Macrophage_M2', 'Memory_B', 'NK_dim', 'Naive_B', 'NonClassical_Mono', 'Plasma_Cell', 'pDC', 'ILC']
df_tot_ntar['Cell'] = pd.Categorical(df_tot_ntar['Cell'], categories=new_cell_order, ordered=True)
df_tot_ntar = df_tot_ntar.sort_values('Cell')
df_tot_tar['Cell'] = pd.Categorical(df_tot_tar['Cell'], categories=new_cell_order, ordered=True)
df_tot_tar = df_tot_tar.sort_values('Cell')

# Make barplots of IL2 target activated and non-activated cells
fig,ax=plt.subplots(1,1,figsize=(14, 8), dpi=400)
plt.bar(df_tot_tar["Cell"], np.zeros([1,len(df_tot_tar["Cell"])])[0], width=0.9, alpha=0.8, color="darkred", label="IL2 target activated cells")
plt.bar(df_tot_ntar["Cell"], np.zeros([1,len(df_tot_tar["Cell"])])[0], width=0.9, alpha=0.8, color="darkblue", label="IL2 target inactivated cells")
for cell in df_tot_tar["Cell"]:
    if df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"].values[0] < df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"].values[0]:
        plt.bar(cell, df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"], width=0.9, alpha=0.8, color="darkblue")
        plt.bar(cell, df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"], width=0.9, alpha=0.8, color="darkred")
    else:
        plt.bar(cell, df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"], width=0.9, alpha=0.8, color="darkred")
        plt.bar(cell, df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"], width=0.9, alpha=0.8, color="darkblue")
plt.errorbar(df_tot_tar["Cell"], df_tot_tar["% target cells"], df_tot_tar["95_conf"], fmt='none', color='black' , ecolor='black', elinewidth=5, capsize=7, barsabove=True, capthick=3)
plt.errorbar(df_tot_ntar["Cell"], df_tot_ntar["% non-target cells"], df_tot_ntar["95_conf"], fmt='none', color='black' , ecolor='black', elinewidth=5, capsize=7, barsabove=True, capthick=3)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=15)
plt.ylabel('Cell proportion (%)', fontsize=20)
plt.xticks(rotation=90, ha="right")
ax.legend(loc="upper right", fontsize=15)
plt.title("Predicted BEMPEG response", fontsize=20)
plt.savefig('figures/BEMPEG_trials/Perc_cells_all_tumor_x10.pdf', bbox_inches='tight', transparent=True)
plt.close()

In [10]:
# Load data and separate it on 3 groups, non-IL2 target cells and IL2 target activated and unactivated cells
df = pd.read_csv("results/BEMPEG_trials/simulations_BEMPEG_tumor_IL2s_d10.csv")
df_high = df.loc[df["IL0_1"]==1e-4][["Cell_type","RB0","STAT50","Result"]]
df_treatment = df.loc[df["IL0_1"]!=1e-4][["Cell_type","RB0","STAT50","Result"]]
# Separate cells into 3 groups
df_treatment["Abs. activ."] = df_treatment["Result"].values
df_treatment["Activ. pot."] = df_treatment["Result"].values/df_high["Result"].values*100
df_treatment = df_treatment[["Cell_type","RB0","Abs. activ.","Activ. pot."]]
thres = 40 #(df_treatment[df_treatment["Abs. activ."]>df_treatment["Abs. activ."].mean()]["Activ. pot."].mean() // 10) * 10 # As threshold we take the mean activ. potential of 50% of cells with more pSTAT5
df_target = df_treatment.loc[df_treatment["RB0"]>2].loc[df_treatment.loc[df_treatment["RB0"]>2]["Activ. pot."]>thres]
df_notarget = df_treatment.loc[df_treatment["RB0"]>2].loc[df_treatment.loc[df_treatment["RB0"]>2]["Activ. pot."]<=thres]

In [11]:
# Make scatter plot
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
ax.scatter(df_treatment["Abs. activ."].values, df_treatment["Activ. pot."].values, s=80, linewidth=0.8, color="darkgreen",marker='o',alpha = 0.33, label="Non-IL2 target cells")
ax.scatter(df_target["Abs. activ."].values, df_target["Activ. pot."].values, s=80, linewidth=0.8, color="darkred",marker='o',alpha = 0.33,label="IL2 target activated cells")
ax.scatter(df_notarget["Abs. activ."].values, df_notarget["Activ. pot."].values, s=80, linewidth=0.8, color="darkblue",marker='o',alpha = 0.33,label="IL2 target inactivated cells")
plt.legend(loc="upper left", fontsize=20)
plt.xscale("log")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("Predicted BEMPEG response", fontsize=25)
plt.xlabel('Absolute activation (pSTAT5 num.)', fontsize=25)
plt.ylabel('Activation potential (% pSTAT5)', fontsize=25)
plt.savefig('figures/BEMPEG_trials/Plot_tumor_d10.pdf', transparent=True, bbox_inches="tight")
plt.close()

In [12]:
# Get percentages of cells in IL2 target, activated cells
perc_cells = Counter([cell.split("__")[0] for cell in df_treatment["Cell_type"].values])
df = pd.DataFrame()
df["Cell"] = perc_cells.keys()
df["% total cells"] = np.array(list(perc_cells.values()))/len(df_treatment["Cell_type"].values)*100
perc_cells = Counter([cell.split("__")[0] for cell in df_target["Cell_type"].values])
df2 = pd.DataFrame()
df2["Cell"] = perc_cells.keys()
df2["% target cells"] = np.array(list(perc_cells.values()))/len(df_target["Cell_type"].values)*100
df_tot_tar = pd.merge(df, df2, on='Cell', how='outer')
df_tot_tar = df_tot_tar.fillna(0)

# Get percentages of cells in IL2 target, non-activated cells
perc_cells = Counter([cell.split("__")[0] for cell in df_notarget["Cell_type"].values])
df3 = pd.DataFrame()
df3["Cell"] = perc_cells.keys()
df3["% non-target cells"] = np.array(list(perc_cells.values()))/len(df_notarget["Cell_type"].values)*100
df_tot_ntar = pd.merge(df, df3, on='Cell', how='outer')
df_tot_ntar = df_tot_ntar.fillna(0)

In [13]:
# Sample from the full population to get error bars in IL2 target, activated cells 
perc_cells = Counter([cell.split("__")[0] for cell in df_target["Cell_type"].values])
df_t = pd.DataFrame()
df_t["Cell"] = perc_cells.keys()
for i in range(0,250):
    df_target_s = df_target.sample(n=int(df_target.index.size/2))
    perc_cells = Counter([cell.split("__")[0] for cell in df_target_s["Cell_type"].values])
    df_ts = pd.DataFrame()
    df_ts["Cell"] = perc_cells.keys()
    df_ts["% "+str(i)] = np.array(list(perc_cells.values()))/len(df_target_s["Cell_type"].values)*100
    df_t = pd.merge(df_t, df_ts, on='Cell', how='outer')
df_t.set_index('Cell', inplace=True)
df_t = df_t.fillna(0)
df_t["95_conf"] = 1.96*(df_t.var(axis=1)/np.sqrt(len(df_t.columns)))
df_t = df_t[["95_conf"]]
df_tot_tar = pd.merge(df_t, df_tot_tar, on='Cell', how='outer')
df_tot_tar = df_tot_tar.fillna(0)

# Sample from the full population to get error bars in IL2 target, non-activated cells 
perc_cells = Counter([cell.split("__")[0] for cell in df_notarget["Cell_type"].values])
df_nt = pd.DataFrame()
df_nt["Cell"] = perc_cells.keys()
for i in range(0,250):
    df_notarget_s = df_notarget.sample(n=int(df_notarget.index.size/2))
    perc_cells = Counter([cell.split("__")[0] for cell in df_notarget_s["Cell_type"].values])
    df_nts = pd.DataFrame()
    df_nts["Cell"] = perc_cells.keys()
    df_nts["% "+str(i)] = np.array(list(perc_cells.values()))/len(df_notarget_s["Cell_type"].values)*100
    df_nt = pd.merge(df_nt, df_nts, on='Cell', how='outer')
df_nt.set_index('Cell', inplace=True)
df_nt = df_nt.fillna(0)
df_nt["95_conf"] = 1.96*(df_nt.var(axis=1)/np.sqrt(len(df_nt.columns)))
df_nt = df_nt[["95_conf"]]
df_tot_ntar = pd.merge(df_nt, df_tot_ntar, on='Cell', how='outer')
df_tot_ntar = df_tot_ntar.fillna(0)

In [14]:
# Get all plots to have the same order of cells
new_cell_order = ['CD4_naive', 'CD4_Central_memory', 'CD4_Effector_memory', 'CD8_Central_memory_cytotoxic', 'CD8_Effector_memory', 'CD8_Exhausted', 'CD8_Exhausted_cytotoxic', 'Classical_Mono', 'ILC1', 'Macrophage_M1', 'NK_bright', 'Tfh', 'Th17', 'Th1', 'Th2', 'Treg_PD1+', 'Treg', 'cDC', 'CD8_Central_memory', 'CD8_Cytotoxic', 'CD8_Effector_memory_cytotoxic', 'CD8_naive', 'ILC2', 'ILC3', 'Macrophage_M2', 'Memory_B', 'NK_dim', 'Naive_B', 'NonClassical_Mono', 'Plasma_Cell', 'pDC', 'ILC']
df_tot_ntar['Cell'] = pd.Categorical(df_tot_ntar['Cell'], categories=new_cell_order, ordered=True)
df_tot_ntar = df_tot_ntar.sort_values('Cell')
df_tot_tar['Cell'] = pd.Categorical(df_tot_tar['Cell'], categories=new_cell_order, ordered=True)
df_tot_tar = df_tot_tar.sort_values('Cell')

# Make barplots of IL2 target activated and non-activated cells
fig,ax=plt.subplots(1,1,figsize=(14, 8), dpi=400)
plt.bar(df_tot_tar["Cell"], np.zeros([1,len(df_tot_tar["Cell"])])[0], width=0.9, alpha=0.8, color="darkred", label="IL2 target activated cells")
plt.bar(df_tot_ntar["Cell"], np.zeros([1,len(df_tot_tar["Cell"])])[0], width=0.9, alpha=0.8, color="darkblue", label="IL2 target inactivated cells")
for cell in df_tot_tar["Cell"]:
    if df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"].values[0] < df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"].values[0]:
        plt.bar(cell, df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"], width=0.9, alpha=0.8, color="darkblue")
        plt.bar(cell, df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"], width=0.9, alpha=0.8, color="darkred")
    else:
        plt.bar(cell, df_tot_tar.loc[df_tot_tar["Cell"]==cell,"% target cells"], width=0.9, alpha=0.8, color="darkred")
        plt.bar(cell, df_tot_ntar.loc[df_tot_ntar["Cell"]==cell,"% non-target cells"], width=0.9, alpha=0.8, color="darkblue")
plt.errorbar(df_tot_tar["Cell"], df_tot_tar["% target cells"], df_tot_tar["95_conf"], fmt='none', color='black' , ecolor='black', elinewidth=5, capsize=7, barsabove=True, capthick=3)
plt.errorbar(df_tot_ntar["Cell"], df_tot_ntar["% non-target cells"], df_tot_ntar["95_conf"], fmt='none', color='black' , ecolor='black', elinewidth=5, capsize=7, barsabove=True, capthick=3)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=15)
plt.ylabel('Cell proportion (%)', fontsize=20)
plt.xticks(rotation=90, ha="right")
ax.legend(loc="upper right", fontsize=15)
plt.title("Predicted BEMPEG response", fontsize=20)
plt.savefig('figures/BEMPEG_trials/Perc_cells_all_tumor_d10.pdf', bbox_inches='tight', transparent=True)
plt.close()